# 1. Raw Data Audit & Structural Validation

## Objective

Before performing any transformation or modeling, this notebook validates the structural integrity of the raw transaction dataset.

The purpose is to:
- Verify schema consistency
- Inspect time coverage
- Check missing values and duplicates
- Confirm dataset suitability for churn modeling

---

In [6]:
import sys
import os
sys.path.append(os.path.abspath(".."))

import pandas as pd
from src.config import RAW_DATA_PATH

In [7]:
df = pd.read_csv(RAW_DATA_PATH, encoding="ISO-8859-1")
df.head()

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom


In [8]:
print("Shape:", df.shape)
df.info()

Shape: (1067371, 8)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1067371 entries, 0 to 1067370
Data columns (total 8 columns):
 #   Column       Non-Null Count    Dtype  
---  ------       --------------    -----  
 0   Invoice      1067371 non-null  object 
 1   StockCode    1067371 non-null  object 
 2   Description  1062989 non-null  object 
 3   Quantity     1067371 non-null  int64  
 4   InvoiceDate  1067371 non-null  object 
 5   Price        1067371 non-null  float64
 6   Customer ID  824364 non-null   float64
 7   Country      1067371 non-null  object 
dtypes: float64(2), int64(1), object(5)
memory usage: 65.1+ MB


## Dataset Overview

The dataset contains ~1 million transaction-level records across ~6,000 customers over a two-year period.

Each row represents a product-level transaction within an invoice.

Key columns:
- Invoice
- StockCode
- Quantity
- InvoiceDate
- Price
- Customer ID
- Country


In [9]:
df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"])

print("Start Date:", df["InvoiceDate"].min())
print("End Date:", df["InvoiceDate"].max())

Start Date: 2009-12-01 07:45:00
End Date: 2011-12-09 12:50:00


In [10]:
print("Unique customers:", df["Customer ID"].nunique())
print("Unique invoices:", df["Invoice"].nunique())
print("Avg transactions per customer:",
      df.groupby("Customer ID")["Invoice"].nunique().mean())

Unique customers: 5942
Unique invoices: 53628
Avg transactions per customer: 7.552339279703803


In [11]:
df.isnull().sum()

Invoice             0
StockCode           0
Description      4382
Quantity            0
InvoiceDate         0
Price               0
Customer ID    243007
Country             0
dtype: int64

In [12]:
df.duplicated().sum()

np.int64(34335)

# Raw Data Audit Summary

## Dataset Overview

- **Total Rows:** 1,067,371  
- **Total Columns:** 8  
- **Memory Usage:** ~65 MB  
- **Unique Customers:** 5,942  
- **Unique Invoices:** 53,628  
- **Average Transactions per Customer:** ~7.55  

The dataset contains transaction-level retail data spanning multiple purchases per customer, making it suitable for behavioral churn modeling.

---

## Time Coverage

- **Start Date:** 2009-12-01  
- **End Date:** 2011-12-09  
- **Total Duration:** ~2 years  

The multi-year coverage provides sufficient temporal depth for:

- Recency analysis  
- Interpurchase time modeling  
- Revenue trend estimation  
- Seasonality feature engineering  

---

## Missing Values

| Column        | Missing Count |
|---------------|--------------|
| Description   | 4,382 |
| Customer ID   | 243,007 |
| Other Columns | 0 |

- Missing **Customer ID** rows represent anonymous transactions and will be removed during preprocessing.
- Missing **Description** is not critical for churn modeling and will not affect feature engineering.

---

## Duplicate Records

- **Duplicate Rows:** 34,335  

Duplicate records will be addressed during the preprocessing stage to ensure data integrity.

---

## Modeling Suitability Assessment

This dataset is appropriate for churn modeling because:

- It contains transaction-level data.
- Customers have multiple historical purchases.
- The time span is sufficient for defining inactivity-based churn.
- Behavioral feature engineering (RFM, purchase gaps, revenue trends, seasonality) is feasible.

### Conclusion

The dataset is structurally sound and sufficiently rich for advanced churn prediction modeling.